#### CPSC excelt to RDF files using OntoSearcher
This notebook is going to be walking through how to use Onto Searcher on the CPSC Compiled Nanoproducts inventory. You can either choose to use the OntoSearcher tutorial or you can use this as a guide because I will try to include some of the things that I have learned regarding using OntoSearcher as someone who knows nothing about nanomaterials

key things: your project structure should look like this 
- 1. A csv folder which will have subfolders for each csv file that we have, we will convert each csv file to RDF and then try to the merge them 
- 2. The ontosearch package which should be given to you
- 3. A folder called owlfiles which stores the different ontologies you want to map on your data: we will be using common ones like enanomapper and npo but depending on the data you might want to select more relevant ones regarding your domain
- 4. Your .json file that has the mapped ontologies, if not look into the onto searcher tutorial notebook but I will try to redo some of the things here as well

**Regarding specific ontologies: some do not work so just try to see what ones are able to match**

In [ ]:
import pandas as pd

In [ ]:
import pandas as pd
df = pd.read_csv('./cpsc_csv/Database/Database.csv')
for col in df.columns:
    print(set(df[col]))
    print()

In [ ]:
df

In [ ]:
import pandas as pd
import os

In [ ]:
import pandas as pd
import os

def clean_folder_name(name):
    """Clean string to be used as a folder name."""
    # Remove invalid characters and replace spaces with underscores
    invalid_chars = '<>:"/\\|?*'
    for char in invalid_chars:
        name = name.replace(char, '')
    return name.strip().replace(' ', '_')

def split_excel_to_folders(excel_file, output_dir):
    """
    Split each sheet of an Excel file into separate folders with CSVs.
    
    Args:
        excel_file (str): Path to the Excel file
        output_dir (str): Directory where folders and CSV files will be saved
    """
    # Create main output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Read all sheets
    excel = pd.ExcelFile(excel_file)
    
    # Process each sheet
    for sheet_name in excel.sheet_names:
        # Create clean folder name
        folder_name = clean_folder_name(sheet_name)
        
        # Create folder path
        folder_path = os.path.join(output_dir, folder_name)
        os.makedirs(folder_path, exist_ok=True)
        
        # Read the sheet
        df = pd.read_excel(excel_file, sheet_name=sheet_name)
        
        # Create CSV filename
        csv_filename = f"{folder_name}.csv"
        csv_path = os.path.join(folder_path, csv_filename)
        
        # Save to CSV
        df.to_csv(csv_path, index=False)
        print(f"Saved {csv_filename} in folder {folder_name}")

if __name__ == "__main__":
    # Input Excel file
    excel_file = "./csvs/CPSC Compiled Nano Products Inventory  7.17.19.xlsm"
    
    # Output directory
    output_dir = "cpsc_csv"
    
    # Run the conversion
    split_excel_to_folders(excel_file, output_dir)

In [ ]:
from ontosearch.onto import ontolister, ontocontext
from ontosearch.csv_importer import load_data
from ontosearch.find import matcher
from ontosearch.onto_api import bioportal_search, unpack_superclass
from ontosearch.rdf_print import term_lookup
from json import dump

In [ ]:
from rdflib import Graph

g = Graph()
g.parse("./owlfiles/materialsmine.ttl", format="turtle")
g.serialize(destination="materialsmine.owl", format="xml")

# try and convert kb bio 

In [ ]:
database_csv = load_data("./cpsc_csv/Database/Database.csv")

In [ ]:
onto = ontolister(
 onto_dir = ('./owlfiles/')
)

In [ ]:
match, unmatch = matcher(onto, database_csv)

In [ ]:
apiKey = '12df8fd3-39f3-4fca-901e-6fc1bd466ab0'
bioapi = bioportal_search(unmatch, apiKey, em=True, mode='col')
unpack_superclass(bioapi, apiKey)


In [ ]:
finalOnto = ontolister(ontofunc = ontocontext,
 onto_dir = ('owlfiles/')
)

In [ ]:
with open('full_onto_cpsc1.json', 'w') as file:
    dump(finalOnto, file)

In [ ]:
match, unmatch = matcher(onto, database_csv)
csv2 = load_data("./cpsc_csv/Nanomaterial/Nanomaterials.csv")
csv3 = load_data("./cpsc_csv/ProductCategories/ProductCategories.csv")
csv4 = load_data("./cpsc_csv/ProductCategories/ProductCategories.csv")
bioapi = bioportal_search(unmatch, apiKey, em=True, mode='col')
unpack_superclass(bioapi, apiKey)

In [ ]:
from ontosearch.onto import ontolister, ontocontext
from ontosearch.csv_importer import load_data
from ontosearch.find import matcher
from ontosearch.onto_api import bioportal_search, unpack_superclass
from ontosearch.onto_api import bioportal_sample, dict_samp, bio_summary
from ontosearch.rdf_print import table_from_file, term_editor, term_lookup
from ontosearch.rdf_print import basic_rdf, relational_rdf_loader
from ontosearch.rdf_print import primenode, node_one, node_two, multi_editor
# supporting libraries
from rdflib import Graph, URIRef, Literal, BNode
import pandas as pd
import json

In [ ]:
with open('./full_onto_cpsc1.json', 'r', encoding='utf-8') as f:
    onto = json.load(f)
for x in onto:
    print(x)
data_match, data_unmatch = matcher(onto, database_csv, context=True)

In [ ]:
for x in onto:
    print(x)

In [ ]:
cpscGraph = Graph()

In [ ]:
import pandas as pd
import re

def sanitize_category(text: str) -> str:
    """
    Sanitize product category names
    """
    # Convert to string in case we get a float or int
    text = str(text)
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove parentheses and their contents
    text = re.sub(r'\(.*?\)', '', text)
    
    # Replace slashes and hyphens with spaces
    text = re.sub(r'[/\-]', ' ', text)
    
    # Remove any remaining special characters
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # Replace multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)
    
    # Trim whitespace
    text = text.strip()
    text = text.replace(' ', '_')
    
    return text

# Read the CSV file
df = pd.read_csv('./cpsc_csv/Database/Database.csv')

# Show some examples before sanitization
print("Before sanitization (sample):")
print(df['Product Category'].head(), "\n")

# Sanitize the Product Category column in place
df['Product Category'] = df['Product Category'].apply(sanitize_category)
df.to_csv('./cpsc_csv/DatabaseSanitized/DatabaseSanitized.csv', index=False)

In [ ]:
database_csv = load_data("./cpsc_csv/DatabaseSanitized/")
database_dfs = table_from_file("./cpsc_csv/DatabaseSanitized/")

In [ ]:
data_match, data_unmatch = matcher(onto, database_csv, context=True)

In [ ]:
cpscGraph = Graph()

In [ ]:
data_match[0]

In [ ]:
basic_rdf(data_match[0], cpscGraph)

# load relational RDF relationships
# this function reads the dataframe structure, and organizes the data
# by providing each row a unique IRI. All other data is organized as
# properties of that row IRI.
relational_rdf_loader(database_dfs, data_match[0], cpscGraph)

# Ontology Reference

| Abbreviation | Base URI | Full Name & Description |
|-------------|----------|------------------------|
| `BAO` | `http://www.bioassayontology.org/bao#` | **BioAssay Ontology** - A semantic description of bioassays and screening results |
| `BERO` | `http://purl.bioontology.org/ontology/BERO/` | **Biomedical and Environmental Research Ontology** - Describes concepts in biomedical and environmental research |
| `CHEAR` | `http://purl.obolibrary.org/obo/CHEAR_` | **Children's Health Exposure Analysis Resource Ontology** - Focused on children's environmental health research and exposure analysis |
| `CHV` | `http://purl.bioontology.org/ontology/CHV/` | **Consumer Health Vocabulary** - Bridges lay and professional medical terminology |
| `DIDEO` | `http://purl.obolibrary.org/obo/DIDEO_` | **Drug-drug Interaction and Evidence Ontology** - Represents drug-drug interactions and their evidence |
| `ENM` | `http://purl.enanomapper.org/onto/ENM_` | **eNanoMapper Ontology** - Describes concepts in nanotechnology safety assessment and characterization |
| `MaterialsMine` | `http://materialsmine.org/ns/` | **Materials Mine Ontology** - Represents materials science and engineering knowledge |
| `MELO` | `http://purl.bioontology.org/ontology/MELO/` | **MELanoma Ontology** - Specialized vocabulary for melanoma research and treatment |
| `NPO` | `http://purl.bioontology.org/ontology/npo#NPO_` | **NanoParticle Ontology** - Represents properties of nanoparticles, their experiments and characterizations |
| `PHARE` | `http://purl.bioontology.org/ontology/PHARE/` | **PHArmacogenomic RElationships Ontology** - Describes relationships between drugs, genes, and clinical outcomes |
| `SCTO` | `http://purl.bioontology.org/ontology/SCTO/` | **Saudi Clinical Trial Ontology** - Standardizes clinical trial terminology and processes |
| `THINF` | `http://purl.bioontology.org/ontology/THINF/` | **Thesaurus Inference** - Used for logical inference and relationships between concepts |

In [ ]:
def nodebag(main_parameter_info, sub_parameter_info, rdfgraph):
    term, termiri = main_parameter_info
    starter_term = sub_parameter_info[0][0]
    # check if 'starter term' already in rdfgraph
    if (None, Literal(starter_term), None) in rdfgraph:
        # make a unique rdf "bag" for each row
        for s, p, o in rdfgraph.triples((None, Literal(starter_term), None)):

            # create unique bag for row
            rowbag = BNode()

            # <row> <parameter> <bag>
            rdfgraph.add((s, URIRef(termiri), rowbag))

            # bag description
            bagid = Literal(f"{term} data for entry: {s} ")
            rdfgraph.add((rowbag, URIRef('http://purl.org/dc/elements/1.1/description'), bagid))

            # <relation> <label> 'label'
            rdfgraph.add((URIRef(termiri), URIRef('http://www.w3.org/2000/01/rdf-schema#label'), Literal(term)))
            
            #NEW TO V2 - nested the entire below block within the bag-creation loop, instead of in line with the overarching if
            #before, we would make a bag for every row before filling any of them
            #with this change, we now make a bag, fill it, and then move on to the next bag
            
            # stuff the bag, looping thru parameter data
            for column, iri, label in sub_parameter_info:
                # add: <column_iri> rdfs:lable 'label'
                rdfgraph.add((URIRef(iri), URIRef('http://www.w3.org/2000/01/rdf-schema#label'), Literal(label)))
                # find the unique bag for each row
                #EDIT - below was original
                #for row, parameter, bag in rdfgraph.triples((None, URIRef(termiri), None)):
                #this variant should work when there's more than one bag per row (i.e. SD2)
                for row, parameter, bag in rdfgraph.triples((None, URIRef(termiri), rowbag)):
                    # loop thru all the pre-existing column relationships
                    for s, p, o in rdfgraph.triples((row, Literal(column), None)):
                        # add: <bag> <column_iri> <data>
                        rdfgraph.add((bag, URIRef(iri), o))
                        # remove: old <row> <column> <data>
                        rdfgraph.remove((s, p, o))


In [ ]:
# First define database and ontology mappings
database_iri = 'http://purl.bioontology.org/ontology/npo#NPO_199'  # Nanotechnology product database

database_edits = [
    ('Product ID', 'http://sbmi.uth.tmc.edu/ontology/ochv#44104'),
    ('Product Subcategory', 'http://semanticscience.org/resource/SIO_001353'),
    ('Nanomaterial Type', 'http://purl.bioontology.org/ontology/npo#NPO_199'),
    ('Reason Nano', 'http://purl.obolibrary.org/obo/CHEBI_33232'),
    ('Manufacturers', 'http://purl.obolibrary.org/obo/OBI_0000835'),
    ('Websites', 'http://semanticscience.org/resource/SIO_000296'),
    ('Countries of Origin', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0034030'),
    ('Testing', 'http://sbmi.uth.tmc.edu/ontology/ochv#12118'),
    ('Notes', 'http://semanticscience.org/resource/SIO_000136'),
    ('Children\'s Use (1-3)', 'http://hadatac.org/ont/chear#PersonalProductUse'),
    ('Exposure (1-10)', 'http://www.ebi.ac.uk/efo/EFO_0000487'),
    ('Toxicity (1-10)', 'http://purl.bioontology.org/ontology/npo#NPO_1338'),
    ('Public Perception (1-5)', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0034030'),
    ('Stakeholder Perception (1-5)', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0036524'),
    ('Relative Level of Concern (RLC)', 'http://purl.obolibrary.org/obo/MESH_D012306'),
    ('Prioritization Score Using Tool', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0549179'),
]

# First add "is a" relationships
for s, p, o in cpscGraph.triples((None, Literal('Product ID'), None)):
    # add <row> a <material>
    cpscGraph.add((s, URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), URIRef(database_iri)))
    # add <material_iri> <rdfs:label> "material"
    cpscGraph.add((URIRef(database_iri), URIRef('http://www.w3.org/2000/01/rdf-schema#label'), Literal('product')))

# Then handle the database edits
for column, iri in database_edits:
    for s, p, o in cpscGraph.triples((None, Literal(column), None)):
        cpscGraph.add((s, URIRef(iri), o))
        cpscGraph.remove((s, p, o))

# After that, create the nodebags
perception_term = ('perception', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0034030')
perception_rdf = [
    ('Public Perception (1-5)', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0034030', 'public perception'),
    ('Stakeholder Perception (1-5)', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0036524', 'stakeholder perception'),
]
nodebag(perception_term, perception_rdf, cpscGraph)

concern_term = ('concern', 'http://purl.obolibrary.org/obo/MESH_D012306')
concern_rdf = [
    ('Relative Level of Concern (RLC)', 'http://purl.obolibrary.org/obo/MESH_D012306', 'risk level'),
    ('Prioritization Score Using Tool', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0549179', 'priority score'),
    ('Exposure (1-10)', 'http://www.ebi.ac.uk/efo/EFO_0000487', 'exposure'),
    ('Toxicity (1-10)', 'http://purl.bioontology.org/ontology/npo#NPO_1338', 'toxicity'),
]
nodebag(concern_term, concern_rdf, cpscGraph)


Ontology mappings per column 

IMPORTANT: Product Subcategory needs manual curation -> 

IMPORTANT: Nanomaterial Type needs manual curation ->  
IMPORTANT: Reason Nano needs manual curation -> http://purl.bioontology.org/ontology/npo#NPO_1490, NPO_199
IMPORTANT: Manufacturers needs manual curation -> 
IMPORTANT: Websites needs manual curation -> 
IMPORTANT: Countries of Origin needs manual curation -> 
IMPORTANT: Children's Use (1-3) needs manual curation -> 
IMPORTANT: Exposure (1-10) needs manual curation -> 
IMPORTANT: Toxicity (1-10) needs manual curation -> 
IMPORTANT: Public Perception (1-5) needs manual curation -> 
IMPORTANT: Stakeholder Perception (1-5) needs manual curation -> 
IMPORTANT: Relative Level of Concern (RLC) needs manual curation -> 
IMPORTANT: Prioritization Score Using Tool needs manual curation -> 
 


In [ ]:
cpscGraph.serialize(format='turtle', destination='cpscdata_wip0.ttl')

In [ ]:
# need to fix the term ontology and then also the material id ones

In [7]:
import pandas as pd
from rdflib import Graph, URIRef, Literal, BNode
from ontosearch.csv_importer import load_data
from ontosearch.rdf_print import table_from_file, relational_rdf_loader, term_lookup
from ontosearch.onto import ontolister, ontocontext
from ontosearch.find import matcher
import json

In [2]:
# Create the RDF graph
cpscGraph = Graph()

In [3]:
# First, load and prepare the CSV data by adding an ID column
df = pd.read_csv("./cpsc_csv/DatabaseSanitized/DatabaseSanitized.csv")
df['ProductID'] = [f"PROD_{str(i).zfill(4)}" for i in df.index]  # Create formatted IDs like PROD_0001
df.to_csv("DatabaseSanitized_with_id.csv", index=False)

In [11]:
# Load the prepared CSV using ontosearcher functions
database_csv = load_data("./cpsc_csv/DatabaseSanitized_with_id/")
database_dfs = table_from_file("./cpsc_csv/DatabaseSanitized_with_id/")

In [5]:
# Define database and ontology mappings
database_iri = 'http://purl.bioontology.org/ontology/npo#NPO_199'  # Nanotechnology product database
database_edits = [
    ('ProductID', 'http://purl.org/dc/terms/identifier'),
    ('Product Category', 'http://sbmi.uth.tmc.edu/ontology/ochv#44104'),
    ('Product Subcategory', 'http://semanticscience.org/resource/SIO_001353'),
    ('Nanomaterial Type', 'http://purl.bioontology.org/ontology/npo#NPO_199'),
    ('Reason Nano', 'http://purl.obolibrary.org/obo/CHEBI_33232'),
    ('Manufacturers', 'http://purl.obolibrary.org/obo/OBI_0000835'),
    ('Websites', 'http://semanticscience.org/resource/SIO_000296'),
    ('Countries of Origin', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0034030'),
    ('Testing', 'http://sbmi.uth.tmc.edu/ontology/ochv#12118'),
    ('Notes', 'http://semanticscience.org/resource/SIO_000136'),
    ('Children\'s Use (1-3)', 'http://hadatac.org/ont/chear#PersonalProductUse'),
    ('Exposure (1-10)', 'http://www.ebi.ac.uk/efo/EFO_0000487'),
    ('Toxicity (1-10)', 'http://purl.bioontology.org/ontology/npo#NPO_1338'),
    ('Public Perception (1-5)', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0034030'),
    ('Stakeholder Perception (1-5)', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0036524'),
    ('Relative Level of Concern (RLC)', 'http://purl.obolibrary.org/obo/MESH_D012306'),
    ('Prioritization Score Using Tool', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0549179'),
]

In [9]:
with open('./full_onto_cpsc1.json', 'r', encoding='utf-8') as f:
    onto = json.load(f)
data_match, data_unmatch = matcher(onto, database_csv, context=True)


number unmatched terms in Product Subcategory: 207

RUN: dideo.owl | exact_binary 
matches: 0
threshold value: 90
RUN: npo-2011-12-08_inferred.owl | exact_binary 
matches: 3
threshold value: 90
RUN: bao_complete.owl | exact_binary 
matches: 2
threshold value: 90
RUN: melanoma_ontology_v4.owl | exact_binary 
matches: 0
threshold value: 90
RUN: CHV_SKOS_2019.owl | exact_binary 
matches: 88
threshold value: 90
RUN: ThesaurusInferred.owl | exact_binary 
matches: 10
threshold value: 90
RUN: enanomapper.owl | exact_binary 
matches: 2
threshold value: 90
RUN: phare.owl | exact_binary 
matches: 0
threshold value: 90
RUN: SCTO.owl | exact_binary 
matches: 0
threshold value: 90

number unmatched terms in Nanomaterial Type: 182

RUN: dideo.owl | exact_binary 
matches: 0
threshold value: 90
RUN: npo-2011-12-08_inferred.owl | exact_binary 
matches: 16
threshold value: 90
RUN: bao_complete.owl | exact_binary 
matches: 0
threshold value: 90
RUN: melanoma_ontology_v4.owl | exact_binary 
matches: 0
th

In [12]:
# Load basic relational RDF relationships
relational_rdf_loader(database_dfs, data_match[0], cpscGraph)  # Empty dict since we're handling mappings manually

# First establish "is a" relationships using ProductID
if (None, Literal('ProductID'), None) in cpscGraph:
    for s, p, o in cpscGraph.triples((None, Literal('ProductID'), None)):
        # add <row> a <product>
        cpscGraph.add((s, URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), URIRef(database_iri)))
        # add <database_iri> <rdfs:label> "product"
        cpscGraph.add((URIRef(database_iri), URIRef('http://www.w3.org/2000/01/rdf-schema#label'), Literal('nanotechnology product')))

# Function for implementing nodebag pattern from NKB example
def nodebag(main_parameter_info, sub_parameter_info, rdfgraph):
    term, termiri = main_parameter_info
    starter_term = sub_parameter_info[0][0]
    
    if (None, Literal(starter_term), None) in rdfgraph:
        for s, p, o in rdfgraph.triples((None, Literal(starter_term), None)):
            if (s, URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), URIRef(database_iri)) in rdfgraph:
                # create unique bag for row
                rowbag = BNode()
                # <row> <parameter> <bag>
                rdfgraph.add((s, URIRef(termiri), rowbag))
                # bag description
                bagid = Literal(f"{term} data for entry: {s}")
                rdfgraph.add((rowbag, URIRef('http://purl.org/dc/elements/1.1/description'), bagid))
                # <relation> <label> 'label'
                rdfgraph.add((URIRef(termiri), URIRef('http://www.w3.org/2000/01/rdf-schema#label'), Literal(term)))
                
                # stuff the bag with parameter data
                for column, iri, label in sub_parameter_info:
                    # add label
                    rdfgraph.add((URIRef(iri), URIRef('http://www.w3.org/2000/01/rdf-schema#label'), Literal(label)))
                    
                    # find and add data
                    for row, param, value in rdfgraph.triples((s, Literal(column), None)):
                        rdfgraph.add((rowbag, URIRef(iri), value))
                        rdfgraph.remove((row, param, value))

# Map the basic columns while verifying types
for column, iri in database_edits:
    for s, p, o in cpscGraph.triples((None, Literal(column), None)):
        if (s, URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), URIRef(database_iri)) in cpscGraph:
            if (s, URIRef(iri), o) in cpscGraph:
                print(f"Warning: Triple already exists for {column}")
            else:
                cpscGraph.add((s, URIRef(iri), o))
                cpscGraph.remove((s, p, o))

# Define and implement perception nodebag
perception_term = ('perception', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0034030')
perception_rdf = [
    ('Public Perception (1-5)', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0034030', 'public perception'),
    ('Stakeholder Perception (1-5)', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0036524', 'stakeholder perception'),
]
nodebag(perception_term, perception_rdf, cpscGraph)

# Define and implement concern nodebag
concern_term = ('concern', 'http://purl.obolibrary.org/obo/MESH_D012306')
concern_rdf = [
    ('Relative Level of Concern (RLC)', 'http://purl.obolibrary.org/obo/MESH_D012306', 'risk level'),
    ('Prioritization Score Using Tool', 'http://sbmi.uth.tmc.edu/ontology/ochv#C0549179', 'priority score'),
    ('Exposure (1-10)', 'http://www.ebi.ac.uk/efo/EFO_0000487', 'exposure'),
    ('Toxicity (1-10)', 'http://purl.bioontology.org/ontology/npo#NPO_1338', 'toxicity'),
]
nodebag(concern_term, concern_rdf, cpscGraph)

# Define custom namespace prefixes
namespace_bindings = [
    ('cpsc', URIRef('http://example.org/cpsc/')),  # Base namespace for your data
    ('npo', URIRef('http://purl.bioontology.org/ontology/npo#')),
    ('sio', URIRef('http://semanticscience.org/resource/')),
    ('obo', URIRef('http://purl.obolibrary.org/obo/')),
    ('ochv', URIRef('http://sbmi.uth.tmc.edu/ontology/ochv#')),
    ('efo', URIRef('http://www.ebi.ac.uk/efo/')),
    ('dcterms', URIRef('http://purl.org/dc/terms/')),
    ('dc', URIRef('http://purl.org/dc/elements/1.1/')),
]

# Bind the namespaces
for prefix, namespace in namespace_bindings:
    cpscGraph.bind(prefix, namespace, override=True)

# Serialize the final RDF
cpscGraph.serialize(format="turtle", destination="cpsc_database.ttl")

 IMPORTANT: Product Subcategory needs manual curation
 IMPORTANT: Nanomaterial Type needs manual curation
 IMPORTANT: Reason Nano needs manual curation
 IMPORTANT: Manufacturers needs manual curation
 IMPORTANT: Websites needs manual curation
 IMPORTANT: Countries of Origin needs manual curation
 IMPORTANT: Children's Use (1-3) needs manual curation
 IMPORTANT: Exposure (1-10) needs manual curation
 IMPORTANT: Toxicity (1-10) needs manual curation
 IMPORTANT: Public Perception (1-5) needs manual curation
 IMPORTANT: Stakeholder Perception (1-5) needs manual curation
 IMPORTANT: Relative Level of Concern (RLC) needs manual curation
 IMPORTANT: Prioritization Score Using Tool needs manual curation
 IMPORTANT: ProductID needs manual curation


<Graph identifier=N663a8cfbe0e54d09b3707fd9d93c0d33 (<class 'rdflib.graph.Graph'>)>

In [14]:
import pandas as pd
from rdflib import Graph, URIRef, Literal, BNode, Namespace
from ontosearch.csv_importer import load_data
from ontosearch.rdf_print import table_from_file, relational_rdf_loader

# Create the RDF graph
cpscGraph = Graph()

# Define and bind namespaces
CPSC = Namespace('http://example.org/cpsc/')
NPO = Namespace('http://purl.bioontology.org/ontology/npo#')
SIO = Namespace('http://semanticscience.org/resource/')
OBO = Namespace('http://purl.obolibrary.org/obo/')
OCHV = Namespace('http://sbmi.uth.tmc.edu/ontology/ochv#')
EFO = Namespace('http://www.ebi.ac.uk/efo/')
DCTERMS = Namespace('http://purl.org/dc/terms/')

# Using the exact nodebag implementation from NKB example
def nodebag(main_parameter_info, sub_parameter_info, rdfgraph):
    term, termiri = main_parameter_info
    starter_term = sub_parameter_info[0][0]
    
    # check if 'starter term' already in rdfgraph
    if (None, Literal(starter_term), None) in rdfgraph:
        # make a unique rdf "bag" for each row
        for s, p, o in rdfgraph.triples((None, Literal(starter_term), None)):
            # create unique bag for row
            rowbag = BNode()
            # <row> <parameter> <bag>
            rdfgraph.add((s, URIRef(termiri), rowbag))
            # bag description
            bagid = Literal(f"{term} data for entry: {s}")
            rdfgraph.add((rowbag, URIRef('http://purl.org/dc/elements/1.1/description'), bagid))
            # <relation> <label> 'label'
            rdfgraph.add((URIRef(termiri), URIRef('http://www.w3.org/2000/01/rdf-schema#label'), Literal(term)))
            
            # stuff the bag with parameter data
            for column, iri, label in sub_parameter_info:
                # add label
                rdfgraph.add((URIRef(iri), URIRef('http://www.w3.org/2000/01/rdf-schema#label'), Literal(label)))
                
                # find and add data
                for s, p, o in rdfgraph.triples((None, Literal(column), None)):
                    rdfgraph.add((rowbag, URIRef(iri), o))
                    rdfgraph.remove((s, p, o))

# # Load and prepare CSV data
# df = pd.read_csv("DatabaseSanitized.csv")
# df['ProductID'] = [f"PROD_{str(i).zfill(4)}" for i in df.index]
# df.to_csv("DatabaseSanitized_with_id.csv", index=False)

# Load data using ontosearcher functions
cpsc_csv = load_data("./cpsc_csv/DatabaseSanitized_with_id/")
cpsc_dfs = table_from_file("./cpsc_csv/DatabaseSanitized_with_id/")

# Load relational RDF relationships
relational_rdf_loader(cpsc_dfs, {}, cpscGraph)

# First establish "is a" relationships using ProductID
if (None, Literal('ProductID'), None) in cpscGraph:
    for s, p, o in cpscGraph.triples((None, Literal('ProductID'), None)):
        # add <row> a <product>
        cpscGraph.add((s, URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), NPO.NPO_199))
        # add <database_iri> <rdfs:label> "product"
        cpscGraph.add((NPO.NPO_199, URIRef('http://www.w3.org/2000/01/rdf-schema#label'), Literal('nanotechnology product')))

# Define perception nodebag
perception_term = ('perception', str(OCHV.C0034030))
perception_rdf = [
    ('Public Perception (1-5)', str(OCHV.C0034030), 'public perception'),
    ('Stakeholder Perception (1-5)', str(OCHV.C0036524), 'stakeholder perception'),
]

# Define concern nodebag
concern_term = ('concern', str(OBO.MESH_D012306))
concern_rdf = [
    ('Relative Level of Concern (RLC)', str(OBO.MESH_D012306), 'risk level'),
    ('Prioritization Score Using Tool', str(OCHV.C0549179), 'priority score'),
    ('Exposure (1-10)', str(EFO.EFO_0000487), 'exposure'),
    ('Toxicity (1-10)', str(NPO.NPO_1338), 'toxicity'),
]

# Apply nodebags
nodebag(perception_term, perception_rdf, cpscGraph)
nodebag(concern_term, concern_rdf, cpscGraph)

# Bind namespaces before serializing
cpscGraph.bind('cpsc', CPSC)
cpscGraph.bind('npo', NPO)
cpscGraph.bind('sio', SIO)
cpscGraph.bind('obo', OBO)
cpscGraph.bind('ochv', OCHV)
cpscGraph.bind('efo', EFO)
cpscGraph.bind('dcterms', DCTERMS)

# Serialize the graph
cpscGraph.serialize(format="turtle", destination="cpsc_database2.ttl")

 IMPORTANT: Product Subcategory needs manual curation
 IMPORTANT: Nanomaterial Type needs manual curation
 IMPORTANT: Reason Nano needs manual curation
 IMPORTANT: Manufacturers needs manual curation
 IMPORTANT: Websites needs manual curation
 IMPORTANT: Countries of Origin needs manual curation
 IMPORTANT: Testing needs manual curation
 IMPORTANT: Notes needs manual curation
 IMPORTANT: Children's Use (1-3) needs manual curation
 IMPORTANT: Exposure (1-10) needs manual curation
 IMPORTANT: Toxicity (1-10) needs manual curation
 IMPORTANT: Public Perception (1-5) needs manual curation
 IMPORTANT: Stakeholder Perception (1-5) needs manual curation
 IMPORTANT: Relative Level of Concern (RLC) needs manual curation
 IMPORTANT: Prioritization Score Using Tool needs manual curation
 IMPORTANT: ProductID needs manual curation


<Graph identifier=Nfde93d5a01784855a2c278840828ed38 (<class 'rdflib.graph.Graph'>)>

In [15]:
import pandas as pd
from rdflib import Graph, URIRef, Literal, BNode, Namespace
from ontosearch.csv_importer import load_data
from ontosearch.rdf_print import table_from_file, relational_rdf_loader

# Create the RDF graph
cpscGraph = Graph()

# Define namespaces more comprehensively
CPSC = Namespace('http://example.org/cpsc/')
NPO = Namespace('http://purl.bioontology.org/ontology/npo#')
SIO = Namespace('http://semanticscience.org/resource/')
OBO = Namespace('http://purl.obolibrary.org/obo/')
OCHV = Namespace('http://sbmi.uth.tmc.edu/ontology/ochv#')
EFO = Namespace('http://www.ebi.ac.uk/efo/')
DCTERMS = Namespace('http://purl.org/dc/terms/')
RDF = Namespace('http://www.w3.org/1999/02/22-rdf-syntax-ns#')
RDFS = Namespace('http://www.w3.org/2000/01/rdf-schema#')
NCIT = Namespace('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#')

# Bind all namespaces
cpscGraph.bind('cpsc', CPSC)
cpscGraph.bind('npo', NPO)
cpscGraph.bind('sio', SIO)
cpscGraph.bind('obo', OBO)
cpscGraph.bind('ochv', OCHV) 
cpscGraph.bind('efo', EFO)
cpscGraph.bind('dcterms', DCTERMS)
cpscGraph.bind('rdf', RDF)
cpscGraph.bind('rdfs', RDFS)
cpscGraph.bind('ncit', NCIT)

def nodebag(main_parameter_info, sub_parameter_info, rdfgraph):
    term, termiri = main_parameter_info
    starter_term = sub_parameter_info[0][0]
    
    if (None, Literal(starter_term), None) in rdfgraph:
        for s, p, o in rdfgraph.triples((None, Literal(starter_term), None)):
            rowbag = BNode()
            rdfgraph.add((s, URIRef(termiri), rowbag))
            bagid = Literal(f"{term} data for entry: {s}")
            rdfgraph.add((rowbag, DCTERMS.description, bagid))
            rdfgraph.add((URIRef(termiri), RDFS.label, Literal(term)))
            
            for column, iri, label in sub_parameter_info:
                rdfgraph.add((URIRef(iri), RDFS.label, Literal(label)))
                for row_s, row_p, row_o in rdfgraph.triples((s, Literal(column), None)):
                    rdfgraph.add((rowbag, URIRef(iri), row_o))
                    rdfgraph.remove((row_s, row_p, row_o))

# Load data
cpsc_csv = load_data("./cpsc_csv/DatabaseSanitized_with_id/")
cpsc_dfs = table_from_file("./cpsc_csv/DatabaseSanitized_with_id/")

# Load basic relationships
relational_rdf_loader(cpsc_dfs, {}, cpscGraph)

# Establish product type relationships
if (None, Literal('ProductID'), None) in cpscGraph:
    for s, p, o in cpscGraph.triples((None, Literal('ProductID'), None)):
        cpscGraph.add((s, RDF.type, NPO.NPO_199))
        cpscGraph.add((NPO.NPO_199, RDFS.label, Literal('nanotechnology product')))

# Define perception metrics nodebag
perception_term = ('perception metrics', str(OCHV.C0034030))
perception_rdf = [
    ('Public Perception (1-5)', str(OCHV.C0034030), 'public perception'),
    ('Stakeholder Perception (1-5)', str(OCHV.C0036524), 'stakeholder perception'),
]

# Define risk assessment nodebag 
risk_term = ('risk assessment', str(OBO.MESH_D012306))
risk_rdf = [
    ('Relative Level of Concern (RLC)', str(OBO.MESH_D012306), 'risk level'),
    ('Prioritization Score Using Tool', str(OCHV.C0549179), 'priority score'),
    ('Exposure (1-10)', str(EFO.EFO_0000487), 'exposure level'),
    ('Toxicity (1-10)', str(NPO.NPO_1338), 'toxicity level'),
]

# Define product info nodebag
product_term = ('product information', str(NCIT.C93400))
product_rdf = [
    ('Product Category', str(NCIT.C93400), 'product category'),
    ('Product Subcategory', str(NCIT.C93401), 'product subcategory'),
    ('Nanomaterial Type', str(NPO.NPO_1808), 'nanomaterial type'),
    ('Reason Nano', str(NCIT.C42614), 'application purpose'),
    ('Manufacturers', str(NCIT.C43530), 'manufacturer'),
    ('Countries of Origin', str(NCIT.C25464), 'country of origin')
]

# Apply nodebags
nodebag(perception_term, perception_rdf, cpscGraph)
nodebag(risk_term, risk_rdf, cpscGraph)
nodebag(product_term, product_rdf, cpscGraph)

# Serialize with all bindings preserved
cpscGraph.serialize(format="turtle", destination="cpsc_database0.ttl")

 IMPORTANT: Product Subcategory needs manual curation
 IMPORTANT: Nanomaterial Type needs manual curation
 IMPORTANT: Reason Nano needs manual curation
 IMPORTANT: Manufacturers needs manual curation
 IMPORTANT: Websites needs manual curation
 IMPORTANT: Countries of Origin needs manual curation
 IMPORTANT: Testing needs manual curation
 IMPORTANT: Notes needs manual curation
 IMPORTANT: Children's Use (1-3) needs manual curation
 IMPORTANT: Exposure (1-10) needs manual curation
 IMPORTANT: Toxicity (1-10) needs manual curation
 IMPORTANT: Public Perception (1-5) needs manual curation
 IMPORTANT: Stakeholder Perception (1-5) needs manual curation
 IMPORTANT: Relative Level of Concern (RLC) needs manual curation
 IMPORTANT: Prioritization Score Using Tool needs manual curation
 IMPORTANT: ProductID needs manual curation


<Graph identifier=N1166b7e6089b4ed2b0366bd1e8b18aa2 (<class 'rdflib.graph.Graph'>)>

In [16]:
import pandas as pd
from rdflib import Graph, URIRef, Literal, BNode, Namespace
from ontosearch.csv_importer import load_data
from ontosearch.rdf_print import table_from_file, relational_rdf_loader

# Create the RDF graph
cpscGraph = Graph()

# Define namespaces
CPSC = Namespace('http://example.org/cpsc/')
NPO = Namespace('http://purl.bioontology.org/ontology/npo#')
SIO = Namespace('http://semanticscience.org/resource/')
OBO = Namespace('http://purl.obolibrary.org/obo/')
OCHV = Namespace('http://sbmi.uth.tmc.edu/ontology/ochv#')
EFO = Namespace('http://www.ebi.ac.uk/efo/')
DCTERMS = Namespace('http://purl.org/dc/terms/')
RDF = Namespace('http://www.w3.org/1999/02/22-rdf-syntax-ns#')
RDFS = Namespace('http://www.w3.org/2000/01/rdf-schema#')
NCIT = Namespace('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#')

# Bind all namespaces
prefix_mappings = {
    'cpsc': CPSC,
    'npo': NPO, 
    'sio': SIO,
    'obo': OBO,
    'ochv': OCHV,
    'efo': EFO,
    'dcterms': DCTERMS,
    'rdf': RDF,
    'rdfs': RDFS,
    'ncit': NCIT
}

for prefix, ns in prefix_mappings.items():
    cpscGraph.bind(prefix, ns)

def create_product_node(row, category):
    """Create a node for a product with all its properties"""
    product_uri = URIRef(f"{CPSC}product_{row['ProductID']}")
    
    # Add type 
    cpscGraph.add((product_uri, RDF.type, NPO.NPO_199))
    
    # Add basic properties
    if pd.notna(row['Product Category']):
        cpscGraph.add((product_uri, NCIT.C93400, Literal(row['Product Category'])))
    if pd.notna(row['Product Subcategory']):
        cpscGraph.add((product_uri, NCIT.C93401, Literal(row['Product Subcategory'])))
    if pd.notna(row['Nanomaterial Type']):
        cpscGraph.add((product_uri, NPO.NPO_1808, Literal(row['Nanomaterial Type'])))
    
    # Create perception metrics bag
    perception_bag = BNode()
    cpscGraph.add((product_uri, OCHV.C0034030, perception_bag))
    cpscGraph.add((perception_bag, DCTERMS.description, 
                   Literal(f"perception metrics data for {row['ProductID']}")))
    
    if pd.notna(row['Public Perception (1-5)']):
        cpscGraph.add((perception_bag, OCHV.C0034030, 
                      Literal(row['Public Perception (1-5)'])))
    if pd.notna(row['Stakeholder Perception (1-5)']):
        cpscGraph.add((perception_bag, OCHV.C0036524,
                      Literal(row['Stakeholder Perception (1-5)'])))

    # Create risk assessment bag  
    risk_bag = BNode()
    cpscGraph.add((product_uri, OBO.MESH_D012306, risk_bag))
    cpscGraph.add((risk_bag, DCTERMS.description,
                   Literal(f"risk assessment data for {row['ProductID']}")))
    
    if pd.notna(row['Relative Level of Concern (RLC)']):
        cpscGraph.add((risk_bag, OBO.MESH_D012306, 
                      Literal(row['Relative Level of Concern (RLC)'])))
    if pd.notna(row['Exposure (1-10)']):
        cpscGraph.add((risk_bag, EFO.EFO_0000487,
                      Literal(row['Exposure (1-10)'])))
    if pd.notna(row['Toxicity (1-10)']):
        cpscGraph.add((risk_bag, NPO.NPO_1338,
                      Literal(row['Toxicity (1-10)'])))
        
    # Add product metadata
    if pd.notna(row['Manufacturers']):
        cpscGraph.add((product_uri, NCIT.C43530, Literal(row['Manufacturers'])))
    if pd.notna(row['Countries of Origin']):
        cpscGraph.add((product_uri, NCIT.C25464, Literal(row['Countries of Origin'])))
    if pd.notna(row['Testing']):
        cpscGraph.add((product_uri, OBO.CHMO_0000101, Literal(row['Testing'])))
    if pd.notna(row['Notes']):
        cpscGraph.add((product_uri, DCTERMS.description, Literal(row['Notes'])))

# Load data
df = pd.read_csv("./cpsc_csv/DatabaseSanitized/DatabaseSanitized.csv")
df['ProductID'] = [f'PROD_{str(i).zfill(4)}' for i in df.index]

# Process each product category
for category in df['Product Category'].unique():
    category_df = df[df['Product Category'] == category]
    for _, row in category_df.iterrows():
        create_product_node(row, category)

# Add class labels
cpscGraph.add((NPO.NPO_199, RDFS.label, Literal('nanotechnology product')))
cpscGraph.add((NCIT.C93400, RDFS.label, Literal('product category')))
cpscGraph.add((NCIT.C93401, RDFS.label, Literal('product subcategory')))
cpscGraph.add((NPO.NPO_1808, RDFS.label, Literal('nanomaterial type')))
cpscGraph.add((OCHV.C0034030, RDFS.label, Literal('perception metrics')))
cpscGraph.add((OBO.MESH_D012306, RDFS.label, Literal('risk assessment')))

# Serialize the graph
cpscGraph.serialize(format="turtle", destination="cpsc_database.ttl")

<Graph identifier=N1653fb2675334d138699543b84a1610b (<class 'rdflib.graph.Graph'>)>